# Reconnaissance d'entités nommées avec SpaCy

La documentation est accessible ici: https://spacy.io/api

## Reconnaissance d'entités nommées : année 1964

**Objectif** : trouver les personnes, les organisations et les lieux les plus présents dans les journaux de 1964.

**Méthode** : spaCy repère les entités dans chaque journal. Pour chaque entité, on compte le nombre de journaux où elle apparaît, et on garde les 20 plus présentes.

**Plan en 5 étapes** :

1. Préparer les journaux de 1964
2. Les personnes (PER)
3. Les organisations (ORG)
4. Les lieux (LOC)
5. Conclusion

## Étape 1 : préparer les journaux de 1964


In [2]:
import spacy

# Charger le modèle français : il contient déjà la reconnaissance des entités
nlp = spacy.load("fr_core_news_md")

/home/minasema/tac/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:1174: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


In [3]:
# Test sur une phrase dont on connaît les entités
doc = nlp("Pierre Harmel est arrivé à Bruxelles pour le gouvernement belge.")

In [4]:
# Afficher chaque entité trouvée et son étiquette
for ent in doc.ents:
    print(ent.text, ent.label_)

Pierre Harmel PER
Bruxelles LOC


In [5]:
from collections import Counter
import os

# Depuis tps/tp3, le dossier des journaux est deux niveaux plus haut
txt_path = "../../data/txt"
year = 1964

# Contrôle : le dossier existe-t-il ?
print(os.path.isdir(txt_path))

True


In [6]:
# Liste des fichiers dont le nom contient 1964
txts = [f for f in os.listdir(txt_path) if os.path.isfile(os.path.join(txt_path, f)) and str(year) in f]

In [7]:
print(len(txts), "journaux de", year)

100 journaux de 1964


## Étape 2 : les personnes

Nous cherchons les entités de type **PER**. Pour chaque journal, une personne n'est comptée qu'une fois. Nous affichons ensuite la liste des 20 personnes présentes dans le plus de journaux.

Avant de compter, nous regardons les entités d'un seul journal, pour voir exactement ce que spaCy a repéré.

In [8]:
# Lire un journal et afficher les 30 premières entités avec leur étiquette
with open(os.path.join(txt_path, txts[0]), "r", encoding="utf-8") as f:
    doc = nlp(f.read())

for ent in doc.ents[:30]:
    print(repr(ent.text), ent.label_)

'VENDREDI 3' MISC
'Porter' PER
'Bon Marché' MISC
'Loterie Nationale' ORG
"Tirage du-6 avril'" MISC
'Blouse' LOC
'Vichy' LOC
'ÔO Fr' ORG
'marine mauve' PER
'Fr + cac j eau ® dixièmes 2 Blouse blanche' ORG
'Fr + Ga' ORG
'NON ILLUSTRE' MISC
'Jupe droite blanche' MISC
'Dior 350 Fr + Ga' MISC
'SAvn' LOC
'Blouse' LOC
'Fr' LOC
'Jupe droite en toile de fibranne' MISC
'Fr' LOC
'B' LOC
'I' LOC
'Jlgslc' MISC
'van' PER
'Blouse' LOC
'Fr' LOC
'Gac' PER
'Jupe droite' MISC
'Raylon' PER
'Fr + ca deau' MISC
'UX' LOC


In [9]:
presence_per = Counter()

for txt in txts:
    with open(os.path.join(txt_path, txt), "r", encoding="utf-8") as f:
        doc = nlp(f.read())

    # set() : chaque personne n'est comptée qu'une fois par journal
    personnes = {ent.text for ent in doc.ents if ent.label_ == "PER" and len(ent.text) > 4}
    presence_per.update(personnes)

print(len(presence_per), "personnes différentes")

13692 personnes différentes


In [10]:
for rang, (nom, n) in enumerate(presence_per.most_common(200), start=1):
    print(f"{rang:2}. {nom} : {n} journaux")

 1. Rossel : 30 journaux
 2. Louise : 27 journaux
 3. Ecrire Ag : 21 journaux
 4. - Tél : 21 journaux
 5. A. P. : 20 journaux
 6. A. F. P. : 16 journaux
 7. Albert : 15 journaux
 8. Reine : 14 journaux
 9. F. Tél : 13 journaux
10. Johnson : 13 journaux
11. Bonne : 12 journaux
12. Martin : 12 journaux
13. F. T. : 12 journaux
14. Molière : 11 journaux
15. A.F.P. : 11 journaux
16. Mozart : 11 journaux
17. Brugmann : 10 journaux
18. Beethoven : 10 journaux
19. Janssens : 10 journaux
20. Président : 9 journaux
21. Shakespeare : 9 journaux
22. Ag Rossel : 9 journaux
23. Couronne : 9 journaux
24. Belliard : 8 journaux
25. Louis XV : 8 journaux
26. Rosse : 8 journaux
27. Léopold : 7 journaux
28. prince Albert : 7 journaux
29. M. Johnson : 7 journaux
30. Herman : 7 journaux
31. Ec Ag : 7 journaux
32. s.d.b : 7 journaux
33. Monsieur : 7 journaux
34. Bruxel : 7 journaux
35. b. ét : 7 journaux
36. R. T. F. : 7 journaux
37. R. T. B. : 7 journaux
38. Madou : 7 journaux
39. O. N. U. : 7 journaux
40. 

## Étape 2b : nettoyer la liste des personnes

La liste brute contient des entrées qui ne sont pas des personnes. Nous les avons repérées dans la liste de l'étape 2a, et nous notons la raison de chaque retrait :


| Groupe | Exemples | Raison |
|---|---|---|
| Petites annonces | ECRIRE AG, TÉL, F. TÉL, EC AG, CRÉD, ATEL, OCCAS, VEND, ENTRET | abréviations et mots d'annonces, pris pour des noms propres à cause des majuscules |
| Mots ordinaires | BONNE, LUNDI, FAIRE, TEMPS, JARDIN, JARD, OFFRE, LIBRE, TEINT, PART, ABANDON, DAME, MESSIEURS, PROF | mots courants, pris pour des noms |
| Fragments d'OCR | CURR, PARF, CULS, MAGN, GREN, COMPTAB, VEUILLEZ, TRAV, GROS, CTRE, CHAMB, FIXE, TAPIS, PIANO, ROYALE, FONTAINE, TOPS | mots coupés ou déformés par l'OCR |
| Organisations | ROSSEL, AG ROSSEL, AGENCE ROSSEL, ROSSE, MONITEUR, MENSUAL, HERMÈS, DU PONT, EASTMAN, AUSTIN, DIESEL, VARIÉTÉS, THÉÂTRE FLAMAND, HALL, VAUDEVILLE, FEUILLETON | sociétés, journaux, marques, théâtres : étape 3 |
| Lieux | BRUGMANN, LOUISE, MADOU, ANSPACH, BRUXEL, SAINT-TROND | hôpital, avenues, places, villes : étape 4 |
| Titres | PRÉSIDENT, PRÉSIDENT DE LA RÉPUBLIQUE, MONSIEUR, MADAME, PAPE | titres sans nom propre |
| Personnages de fiction | ZORRO, CAVOR, MACBETH | personnages de films ou de pièces |
| Sigles avec un point | A. P., A. F. P., A.F.P., F. T., O.N.U., R.T.B. | sigles et initiales : écartés par la règle « pas de point » |

On enlève aussi le tiret du début (« - Tél ») et on met tout en majuscules, pour que « Monsieur » et « MONSIEUR » soient comptés ensemble.

Nous gardons pour l'instant ALBERT, REINE, JOHNSON, MARTIN, LOUIS et WASHINGTON. Ce sont peut-être des personnes, mais certaines peuvent être des lieux ou des fragments : nous les signalerons dans la conclusion.

In [23]:
# Entrées qui ne sont pas des personnes (raisons : voir le tableau du Markdown)
exclus = {
    "ECRIRE AG", "TÉL", "F. TÉL", "EC AG", "CRÉD", "ATEL", "OCCAS", "VEND", "ENTRET",
    # mots ordinaires
    "BONNE", "LUNDI", "FAIRE", "TEMPS", "JARDIN", "JARD", "OFFRE", "LIBRE", "TEINT",
    "PART", "ABANDON", "DAME", "MESSIEURS", "PROF",
    # fragments d'OCR
    "CURR", "PARF", "CULS", "MAGN", "GREN", "COMPTAB", "VEUILLEZ", "TRAV", "GROS",
    "CTRE", "CHAMB", "FIXE", "TAPIS", "PIANO", "ROYALE", "FONTAINE", "TOPS",
    # organisations
    "ROSSEL", "AG ROSSEL", "AGENCE ROSSEL", "ROSSE", "MONITEUR", "MENSUAL", "HERMÈS",
    "DU PONT", "EASTMAN", "AUSTIN", "DIESEL", "VARIÉTÉS", "THÉÂTRE FLAMAND", "HALL",
    "VAUDEVILLE", "FEUILLETON",
    # lieux
    "BRUGMANN", "LOUISE", "MADOU", "ANSPACH", "BRUXEL", "SAINT-TROND",
    # titres
    "PRÉSIDENT", "PRÉSIDENT DE LA RÉPUBLIQUE", "MONSIEUR", "MADAME", "PAPE",
    # personnages de fiction
    "ZORRO", "CAVOR", "MACBETH",

}

In [24]:
presence_propres = Counter()

for txt in txts:
    with open(os.path.join(txt_path, txt), "r", encoding="utf-8") as f:
        doc = nlp(f.read())

    # 1. Nettoyage : tiret retiré, majuscules
    personnes = {ent.text.strip("- ").upper() for ent in doc.ents if ent.label_ == "PER"}

    # 2. On écarte les noms trop courts, les sigles avec un point, et la liste d'exclusion
    personnes = {p for p in personnes if len(p) > 3 and "." not in p and p not in exclus}

    presence_propres.update(personnes)

print(len(presence_propres), "personnes retenues")

10807 personnes retenues


In [25]:
for rang, (nom, n) in enumerate(presence_propres.most_common(200), start=1):
    print(f"{rang:2}. {nom} : {n} journaux")

 1. ALBERT : 15 journaux
 2. REINE : 14 journaux
 3. JOHNSON : 13 journaux
 4. MARTIN : 12 journaux
 5. MOLIÈRE : 11 journaux
 6. MOZART : 11 journaux
 7. BEETHOVEN : 10 journaux
 8. JANSSENS : 10 journaux
 9. SHAKESPEARE : 9 journaux
10. COURONNE : 9 journaux
11. BACH : 9 journaux
12. BELLIARD : 8 journaux
13. LOUIS XV : 8 journaux
14. LOUIS : 8 journaux
15. LÉOPOLD : 7 journaux
16. PRINCE ALBERT : 7 journaux
17. HERMAN : 7 journaux
18. ROOSEVELT : 7 journaux
19. GÉNÉRAL DE GAULLE : 7 journaux
20. FRANÇOISE : 7 journaux
21. SCHUMANN : 7 journaux
22. BRAHMS : 7 journaux
23. FRANÇOIS : 6 journaux
24. JEAN : 6 journaux
25. GRÉTRY : 6 journaux
26. MARTINE : 6 journaux
27. THOMAS : 6 journaux
28. ANDERL : 6 journaux
29. 4 CHB : 6 journaux
30. CONN : 6 journaux
31. JACOBS : 6 journaux
32. VERHAEGEN : 6 journaux
33. JOSEPH II : 6 journaux
34. VIVALDI : 6 journaux
35. HARDY : 6 journaux
36. JEUNESSE : 6 journaux
37. ERHARD : 6 journaux
38. LÉOPOLD II : 6 journaux
39. WAGNER : 6 journaux
40. F

## Étape 3 : les organisations

Même comptage que pour les personnes, avec les entités de type **ORG**. Nous affichons la liste brute. Le nettoyage viendra ensuite, en suivant la même démarche : on lit la liste, on repère ce qui n'est pas une organisation, et on note la raison de chaque retrait.

In [26]:
presence_org = Counter()

for txt in txts:
    with open(os.path.join(txt_path, txt), "r", encoding="utf-8") as f:
        doc = nlp(f.read())

    organisations = {ent.text for ent in doc.ents if ent.label_ == "ORG" and len(ent.text) > 3}
    presence_org.update(organisations)

print(len(presence_org), "organisations différentes")

4296 organisations différentes


In [29]:
for rang, (nom, n) in enumerate(presence_org.most_common(200), start=1):
    print(f"{rang:2}. {nom} : {n} journaux")

 1. Agence Rossel : 22 journaux
 2. Conseil : 17 journaux
 3. Ford : 16 journaux
 4. Reuter : 15 journaux
 5. Inst : 12 journaux
 6. ALLEMAGNE : 11 journaux
 7. PARC : 11 journaux
 8. Fiat : 11 journaux
 9. LUXEMBOURG : 11 journaux
10. Société : 10 journaux
11. FIAT : 10 journaux
12. Tokio : 10 journaux
13. EMPLOYEE : 10 journaux
14. EUROPE : 9 journaux
15. FRANÇAIS : 9 journaux
16. LEOPOLD : 9 journaux
17. GALERIES : 9 journaux
18. Standard : 9 journaux
19. Téléphone : 9 journaux
20. Galeries : 8 journaux
21. Opéra National : 8 journaux
22. Couronne : 8 journaux
23. Import : 8 journaux
24. FORD : 8 journaux
25. SUETENS : 8 journaux
26. Variétés : 8 journaux
27. Nations Unies : 8 journaux
28. FIRME : 8 journaux
29. GAND : 7 journaux
30. Chambre : 7 journaux
31. Cour : 7 journaux
32. Shell : 7 journaux
33. Opel : 7 journaux
34. OPEL : 7 journaux
35. ministre des Affaires étrangères : 7 journaux
36. PREVISIONS : 7 journaux
37. DACTYLO : 7 journaux
38. Sénat : 7 journaux
39. Parlement : 7

# Nettoyage des organismes

In [30]:
# Entrées qui ne sont pas des organisations (raisons : voir le tableau du Markdown)
exclus_org = {
    # pays, villes, régions : étape 4
    "ALLEMAGNE", "LUXEMBOURG", "EUROPE", "SUISSE", "MONACO", "YOUGOSLAVIE", "TOKIO",
    "GAND", "LOUVAIN", "EVERE", "WAVRE", "ANDERLECHT", "AUDERGHEM", "BERCHEM",
    "HAINAUT", "WALON", "WALLON", "PAYS", "BRISTOL",
    # personnes : étape 2
    "GILSON", "SUETENS", "PIRON", "MEYSMANS", "AERTS", "ALEX", "DOUGLAS",
    # annonces et abréviations
    "E.A.", "EXPÉRIM", "TRAV", "PAIEM", "APPAR", "ACHETETOUT", "ACHETE", "COIFF",
    "PR MÉN", "S.B.", "UNLL", "COU-", "BRU-", "MENAGE", "DEMANDONS", "EXPERIMENTE",
    "VERTICALEMENT", "CAPRICORNE", "GARANTIE", "CRÉD", "CREDIT", "IMPEC", "NORV",
    "PRÉVISIONS", "ASSURANCES",
    # mots ordinaires et fragments
    "EMPLOYEE", "STANDARD", "TÉLÉPHONE", "IMPORT", "SOIR", "FIRME", "GARAGE",
    "DIRECTION", "CENTRE", "CENTRAL", "CENTENAIRE", "INST", "INTER", "INTERNATIONAL",
    "NATIONS", "SOCIÉTÉ", "SOCIETE", "JUSTICE", "DIRECT", "COUPÉ", "REPORT", "ECHO",
    "CLASSEMENT", "LIGUE", "BLANCS", "INTIMITÉ",
    # sigles et fragments
    "DM F", "TÉLÉPHOTO", "PETRO", "MAGNÉTOPH", "BANK",
}

In [32]:
presence_org_propres = Counter()

for txt in txts:
    with open(os.path.join(txt_path, txt), "r", encoding="utf-8") as f:
        doc = nlp(f.read())

    # 1. Nettoyage : tiret retiré, majuscules
    organisations = {ent.text.strip("- ").upper() for ent in doc.ents if ent.label_ == "ORG"}

    # 2. On écarte les noms trop courts, les sigles avec un point, et la liste d'exclusion
    organisations = {o for o in organisations if len(o) > 3 and "." not in o and o not in exclus_org}

    presence_org_propres.update(organisations)

print(len(presence_org_propres), "organisations retenues")

3717 organisations retenues


In [34]:
for rang, (nom, n) in enumerate(presence_org_propres.most_common(200), start=1):
    print(f"{rang:2}. {nom} : {n} journaux")

 1. AGENCE ROSSEL : 22 journaux
 2. FORD : 18 journaux
 3. CONSEIL : 17 journaux
 4. FIAT : 17 journaux
 5. REUTER : 15 journaux
 6. PARC : 11 journaux
 7. FRANÇAIS : 10 journaux
 8. GALERIES : 10 journaux
 9. UNION : 9 journaux
10. LEOPOLD : 9 journaux
11. OPEL : 9 journaux
12. OPÉRA NATIONAL : 8 journaux
13. PEUGEOT : 8 journaux
14. COURONNE : 8 journaux
15. VARIÉTÉS : 8 journaux
16. NATIONS UNIES : 8 journaux
17. CHAMBRE : 7 journaux
18. SHELL : 7 journaux
19. COUR : 7 journaux
20. MINISTRE DES AFFAIRES ÉTRANGÈRES : 7 journaux
21. AGENCE : 7 journaux
22. PREVISIONS : 7 journaux
23. DACTYLO : 7 journaux
24. SÉNAT : 7 journaux
25. PARLEMENT : 7 journaux
26. VOLKSWAGEN : 6 journaux
27. RITZ : 6 journaux
28. GAITE : 6 journaux
29. NOVELTY : 6 journaux
30. VAUDEVILLE : 6 journaux
31. METRO : 6 journaux
32. RACING : 6 journaux
33. BENELUX : 6 journaux
34. PORSCHE : 6 journaux
35. VOLVO : 6 journaux
36. PHILIPS : 6 journaux
37. BELGIAN : 6 journaux
38. UNION SOVIÉTIQUE : 6 journaux
39. COM

## Étape 4 : les lieux

Même comptage, avec les entités de type **LOC** : villes, pays, régions, rues, avenues. Nous affichons la liste brute. Le nettoyage viendra après l'observation de cette liste.

In [35]:
presence_loc = Counter()

for txt in txts:
    with open(os.path.join(txt_path, txt), "r", encoding="utf-8") as f:
        doc = nlp(f.read())

    lieux = {ent.text for ent in doc.ents if ent.label_ == "LOC" and len(ent.text) > 3}
    presence_loc.update(lieux)

print(len(presence_loc), "lieux différents")

11312 lieux différents


In [38]:
for rang, (nom, n) in enumerate(presence_loc.most_common(11312), start=1):
    print(f"{rang:2}. {nom} : {n} journaux")

 1. Bruxelles : 92 journaux
 2. Belgique : 64 journaux
 3. Anvers : 55 journaux
 4. Liège : 53 journaux
 5. Paris : 50 journaux
 6. Etat : 50 journaux
 7. France : 46 journaux
 8. Brux : 41 journaux
 9. Allemagne : 40 journaux
10. Gand : 34 journaux
11. Namur : 32 journaux
12. Europe : 32 journaux
13. Luxembourg : 32 journaux
14. Louvain : 32 journaux
15. Mons : 31 journaux
16. Londres : 30 journaux
17. Téléph : 29 journaux
18. Centre : 29 journaux
19. Etats-Unis : 27 journaux
20. Uccle : 26 journaux
21. BRUXELLES : 25 journaux
22. Schaerbeek : 25 journaux
23. Anderlecht : 24 journaux
24. Ostende : 24 journaux
25. Belge : 23 journaux
26. Rome : 23 journaux
27. Forest : 21 journaux
28. Wavre : 21 journaux
29. Italie : 21 journaux
30. rue Royale : 21 journaux
31. Ixelles : 20 journaux
32. Charleroi : 20 journaux
33. Angleterre : 20 journaux
34. Grande-Bretagne : 20 journaux
35. Suisse : 20 journaux
36. la France : 20 journaux
37. New York : 19 journaux
38. Congo : 19 journaux
39. Bruxell